In [1]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import openpyxl
import re 

import warnings
warnings.filterwarnings("ignore")

In [2]:
start_time = time.time()

In [3]:
bd.projects.set_current('nitrous_oxide_production_monte_carlo')

In [4]:
sorted(list(bd.databases))

['ecoinvent-3.10-biosphere', 'ecoinvent-3.10-cutoff', 'nitrous-oxide-ei310']

In [5]:
methods = [('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')]
len(methods)

1

In [6]:
methods

[('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')]

In [7]:
db = bd.Database('nitrous-oxide-ei310')
acts = [act for act in db if 'hydrogen peroxide production' in act['name'] or 'nitrous oxide production' in act['name'] or 'phenol production' in act['name']]
len(acts)

14

In [8]:
eidb = bd.Database('ecoinvent-3.10-cutoff')
act = [a for a in eidb if 'phenol production, cumene oxidation' in a['name'] and 'phenol' in a['reference product'] and 'RoW' in a['location']][0]
acts = acts + [act]
len(acts)

15

In [9]:
mc_df = pd.DataFrame()

In [10]:
demands = [{act:1} for act in acts]
all_demands = {k: 1 for demand in demands for k in demand}
lca = bc.LCA(demand=all_demands, method=methods[0], use_distributions=True)
lca.lci()

C_matrices = {}
for method in methods:
    lca.switch_method(method)
    C_matrices[method] = lca.characterization_matrix.copy()
    
results = {
    act['name']: [] for act in acts
}

for _ in range(500):
    next(lca)
    for index, demand in enumerate(demands):
        lca.lci({key.id: value for key, value in demand.items()})
        for method in methods:
            name = str(list(demand.keys())[0])
            name = name.replace("dict_keys([", "").replace("])", "").replace("'", "")
            name = name[:name.find(' (')]
            results[name].append(
                (C_matrices[method] * lca.inventory).sum()
            )

for key, value in results.items():
    mc_df[key] = pd.Series(value)

In [11]:
mc_df

,"phenol production, hydrogen peroxide, fossil","phenol production, nitrous oxide, fossil","nitrous oxide production, AON 90 cryogenic, fossil ammonia","nitrous oxide production, BAU, fossil ammonia","nitrous oxide production, AON 90 membrane, fossil ammonia","nitrous oxide production, AON 90 membrane, green ammonia","nitrous oxide production, AON 90 cryogenic, green ammonia","nitrous oxide production, AON 100, fossil ammonia","phenol production, hydrogen peroxide, green","nitrous oxide production, AON 100, green ammonia","hydrogen peroxide production, AO, green hydrogen","hydrogen peroxide production, AO, fossil hydrogen","nitrous oxide production, BAU, green ammonia","phenol production, nitrous oxide, green","phenol production, cumene oxidation"
0,3.604792,4.604262,4.151045,3.177189,2.845185,0.796947,2.096142,2.528808,3.267270,0.673780,0.489089,1.193726,1.149918,3.722567,4.558561
1,3.717569,4.674633,4.221402,3.268271,2.936530,0.931772,2.210121,2.611506,3.318361,0.795857,0.525655,1.359076,1.284035,3.811655,4.986521
2,3.608484,4.772458,4.206624,3.390981,3.074980,1.280357,2.406162,2.737919,3.260600,1.112583,0.737663,1.463935,1.614729,3.999936,4.555404
3,3.412461,4.465890,4.281172,3.430489,3.114830,0.998021,2.157475,2.773660,3.015460,0.856529,0.570086,1.398899,1.335348,3.554677,5.213564
4,3.165817,4.483694,4.939618,3.967015,3.666360,1.156750,2.421841,3.273448,2.868678,1.000569,0.599384,1.219716,1.483094,3.403395,5.104945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,3.320902,4.491481,4.879405,3.896537,3.587735,1.029541,2.312886,3.201748,2.971168,0.884868,0.537842,1.267976,1.364529,3.390267,6.087708
496,3.563110,4.578783,4.266756,3.389999,3.079676,0.820800,2.000530,2.741848,3.241074,0.696052,0.465749,1.138057,1.154245,3.606416,6.410180
497,4.130177,4.993209,4.109930,3.238331,2.939001,1.070111,2.234959,2.615097,3.785686,0.922500,0.591149,1.310336,1.388572,4.188717,5.124975
498,3.007231,4.275055,4.677581,3.769676,3.483051,0.755309,1.940963,3.108219,2.631733,0.637785,0.468021,1.251941,1.069856,3.100857,4.294376


In [12]:
mc_df.to_excel(os.path.join('..', 'results', 'monte_carlo.xlsx'))